In [ ]:
from typing import TypedDict, Annotated, Sequence

from langgraph.constants import START, END
from langgraph.graph import StateGraph
from langgraph.types import Send
from operator import add
from loguru import logger
from rich import print


#1. 状態を宣言
#1.1 グローバル状態を宣言
class OverAllState(TypedDict):
    input_values: list[str]
    entries: Annotated[list[tuple[str, int]], add]
    word_counts: dict[str, int]


class MaperInputState(TypedDict):
    input_value: str


#2. ノードを宣言
#2.1 ディスパッチノード
def router_node(state: OverAllState) -> Sequence[Send]:
    input_values = state["input_values"]
    task = []
    for input_value in input_values:
        task.append(
            Send("mapper_node", {"input_value": input_value})
        )
    return task


#2.2 個々の文を受け取り、単語に分割し、タプルに組み立ててグローバル状態に格納する
def mapper_node(state: MaperInputState) -> OverAllState:
    input_value = state["input_value"]
    words = input_value.split(" ")
    #  (hello, 1), (world, 1)
    entries = []
    for word in words:
        entries.append((word, 1))
    return {
        "entries": entries
    }


#2.3 すべての mapper_node が分割したタプルを集約してマージする
def reducer_node(state: OverAllState) -> OverAllState:
    entries = state["entries"]
    shuffle_dict = {}
    # hello   -> [1, 1, 1]
    for k, v in entries:
        if k not in shuffle_dict:
            shuffle_dict[k] = [v]
        else:
            shuffle_dict[k].append(v)

    logger.info("reducer shuffle entries:{}", shuffle_dict)
    reduce_dict = {}

    for k, v in shuffle_dict.items():
        reduce_dict[k] = len(v)

    logger.info("reducer {}", reduce_dict)

    return {
        "word_counts": reduce_dict
    }


#3. グラフを構築
builder = StateGraph(state_schema=OverAllState)

builder.add_node("mapper_node", mapper_node)
builder.add_node("reducer_node", reducer_node)

builder.add_conditional_edges(START, router_node, path_map=["mapper_node"])
builder.add_edge("mapper_node", "reducer_node")
builder.add_edge("reducer_node", END)

graph = builder.compile()

res = graph.invoke({
    "input_values": ["hello world", "hello atguigu", "hello llm"]
})
print(res)

from IPython.display import display

display(graph)

